
# N08: High-Dimensional Manifold Engineering
**Objective:** Breach the 0.9498 baseline (without pseudo-labeling leakage) by projecting continuous variables into discrete manifolds using K-Means clustering, generating missingness indicators, and explicitly crossing 3-dimensional categorical interactions based on latest community consensus.


In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 2026
N_FOLDS = 5


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Generate Missingness Indicators BEFORE Imputation
missing_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'water_intake', 'stress_level', 'sleep_quality']
for df in [train_df, test_df]:
    for col in missing_cols:
        if df[col].isnull().sum() > 0:
            df[f'{col}_is_missing'] = df[col].isnull().astype(int).astype(str)

# Feature Engineering: Bins and Interactions
for df in [train_df, test_df]:
    # Sleep Bin
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    
    # 2D Interaction
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']
    
    # 3D Interaction: Stress x Sleep x Physical Activity
    df['stress_sleep_activity_interact'] = df['stress_sleep_interact'] + '_' + df['physical_activity_level'].astype(str)

# Define Base Column Types
cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 
            'stress_sleep_interact', 'stress_sleep_activity_interact']
# Add missingness flags to cat_cols
missing_flags = [c for c in train_df.columns if '_is_missing' in c]
cat_cols.extend(missing_flags)

num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])

X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

# Categorical Null Handling
for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

# Numeric Imputation
num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Continuous Manifold Projection (K-Means)
print("--- Generating K-Means Clustering Manifolds ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num_raw)
X_test_scaled = scaler.transform(X_test_num_raw)

# We fit KMeans strictly on the training set to prevent data leakage, then project onto test set
kmeans = KMeans(n_clusters=15, random_state=SEED, n_init=10)
train_clusters = kmeans.fit_predict(X_train_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# Add cluster assignments as a new categorical feature
X_train_raw['kmeans_cluster'] = train_clusters.astype(str)
X_test_raw['kmeans_cluster'] = test_clusters.astype(str)

cat_cols.append('kmeans_cluster')
print(f"K-Means clusters added. Total categorical features: {len(cat_cols)}")


In [ ]:

# 3. Model Training: Target Encoded CatBoost
oof_preds = np.zeros(len(train_df))
tst_probs = np.zeros((len(test_df), 3))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("\n--- Phase 3: Evaluating High-Dimensional Manifold ---")

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    
    m = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=6, class_weights=weights, random_seed=SEED+fold, verbose=0, task_type="CPU")
    m.fit(X_tr_full, y_tr, eval_set=(X_va_full, y_va), early_stopping_rounds=100)
    
    val_preds = m.predict(X_va_full).flatten()
    oof_preds[va_i] = val_preds
    tst_probs += m.predict_proba(X_te_full) / N_FOLDS
    
    print(f"  Fold {fold + 1}/{N_FOLDS} completed. Balanced Acc: {balanced_accuracy_score(y_va, val_preds):.5f}")

final_score = balanced_accuracy_score(y_train, oof_preds)
print(f"\nFinal Manifold OOF Balanced Accuracy: {final_score:.5f}")


In [ ]:

# 4. Generate Predictions
sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in tst_probs.argmax(axis=1)]
})

sub_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")
